# Candle Scan Operations Demo (Rust / evcxr)

This notebook demonstrates **scan/prefix-sum operations** using Candle tensor operations.

? **Scan Operations**: Explore parallel prefix algorithms:
- **Cumulative Sum (cumsum)**: Inclusive prefix sum along dimensions
- **Inclusive Scan**: Work-efficient scan implementation (CUDA optimized)
- **Exclusive Scan**: Prefix sum with zero padding
- **Multi-dimensional**: Row-wise and column-wise scans

⚡ **Performance Features**:
- GPU acceleration when available
- Memory-efficient implementations
- Parallel processing support
- Large tensor optimizations

🎯 **Use Cases**: Learn practical applications:
- Running totals and statistics
- Parallel algorithms (sorting, etc.)
- Image processing (integral images)
- Scientific computing workflows

These are clean demos; for full exploration see `research/notebooks/`.

In [ ]:
// Environment Setup for Scan Operations
// Configure dependencies and workspace for tensor scanning

// Core dependencies for scan operations
:dep candle-core = { path = "../../candle-core", default-features = false }
:dep anyhow = "1" 
:dep candle-notebooks = { path = "../research/notebooks/candle_notebooks" }

// Import required modules for scan operations
use candle_core::{Device, Tensor, DType, D};
use candle_notebooks as nb;

// Initialize workspace for scan operations
nb::set_notebook_cwd().unwrap();
nb::set_image_store_rel_dir("images_store").unwrap();
std::fs::create_dir_all("images_store").ok();

let device = Device::Cpu;
println!("📊 Scan Operations Environment Ready!");
println!("  Device: {:?}", device);
println!("  Tensor scanning workspace initialized");
println!("  Ready for prefix-sum operations");

Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION
✓ Notebook initialized successfully!
  Working directory: "/home/rustuser/projects/rust/from_github/candle/0aEXPLORATION"
  Device: Cpu
  Location: demos/scan_operations_demo.rs.ipynb
  Paths are resilient to notebook file moves within the candle repository.


Saving images to: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/images_store

In [3]:
// Basic 1D Scan Operations with Candle
// Demonstrates cumulative sum, inclusive scan, and exclusive scan

// Create a simple 1D tensor for demonstration
let data = Tensor::new(&[1.0f32, 2.0, 3.0, 4.0, 5.0], &device)?;
println!("Input tensor: {:?}", data.to_vec1::<f32>()?);

// Inclusive scan (cumulative sum) - most common operation
let cumsum_result = data.cumsum(D::Minus1)?;
println!("Cumulative sum: {:?}", cumsum_result.to_vec1::<f32>()?);

// Inclusive scan (alternative API - same result as cumsum)
let inclusive_result = data.inclusive_scan(D::Minus1)?;
println!("Inclusive scan: {:?}", inclusive_result.to_vec1::<f32>()?);

// Exclusive scan (shifted with zero padding)
let exclusive_result = data.exclusive_scan(D::Minus1)?;
println!("Exclusive scan: {:?}", exclusive_result.to_vec1::<f32>()?);

// Verify the relationship: exclusive[i+1] = inclusive[i]
println!("\n✓ Scan operations completed successfully!");
println!("  Expected cumsum: [1.0, 3.0, 6.0, 10.0, 15.0]");
println!("  Expected exclusive: [0.0, 1.0, 3.0, 6.0, 10.0]");

Input tensor: [1.0, 2.0, 3.0, 4.0, 5.0]
Cumulative sum: [1.0, 3.0, 6.0, 10.0, 15.0]
Inclusive scan: [1.0, 3.0, 6.0, 10.0, 15.0]
Exclusive scan: [0.0, 1.0, 3.0, 6.0, 10.0]

✓ Scan operations completed successfully!
  Expected cumsum: [1.0, 3.0, 6.0, 10.0, 15.0]
  Expected exclusive: [0.0, 1.0, 3.0, 6.0, 10.0]
Cumulative sum: [1.0, 3.0, 6.0, 10.0, 15.0]
Inclusive scan: [1.0, 3.0, 6.0, 10.0, 15.0]
Exclusive scan: [0.0, 1.0, 3.0, 6.0, 10.0]

✓ Scan operations completed successfully!
  Expected cumsum: [1.0, 3.0, 6.0, 10.0, 15.0]
  Expected exclusive: [0.0, 1.0, 3.0, 6.0, 10.0]


// 2D Scan Operations - Row-wise and Column-wise
// Demonstrates scanning along different dimensions

// Create a 2D tensor for multi-dimensional scan operations
let matrix = Tensor::new(&[[1.0f32, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]], &device)?;
println!("Input matrix:");
println!("  {:?}", matrix.to_vec2::<f32>()?);

// Row-wise scan (along last dimension, dim=1)
let row_scan = matrix.cumsum(D::Minus1)?;
println!("\nRow-wise cumsum (along columns):");
println!("  {:?}", row_scan.to_vec2::<f32>()?);

// Column-wise scan (along first dimension, dim=0)
let col_scan = matrix.cumsum(0)?;
println!("\nColumn-wise cumsum (along rows):");
println!("  {:?}", col_scan.to_vec2::<f32>()?);

// Demonstrate exclusive scan in 2D
let row_exclusive = matrix.exclusive_scan(1)?;
println!("\nRow-wise exclusive scan:");
println!("  {:?}", row_exclusive.to_vec2::<f32>()?);

// Different data types
let matrix_i32 = Tensor::new(&[[1i64, 2, 3], [4, 5, 6], [7, 8, 9]], &device)?;
let cumsum_i32 = matrix_i32.cumsum(0)?;
println!("\nInteger cumsum (column-wise):");
println!("  {:?}", cumsum_i32.to_vec2::<i64>()?);

println!("\n✓ 2D scan operations completed successfully!");

In [4]:
// Performance and Advanced Scan Features
// Demonstrates larger tensors and optimization characteristics

use std::time::Instant;

// Large 1D scan performance test
let size = 10000;
let large_data: Vec<f32> = (1..=size).map(|_| 1.0f32).collect();
let large_tensor = Tensor::new(large_data.as_slice(), &device)?;

let start = Instant::now();
let large_cumsum = large_tensor.cumsum(D::Minus1)?;
let duration = start.elapsed();

println!("Large tensor scan performance:");
println!("  Size: {} elements", size);
println!("  Time: {:?}", duration);
println!("  First few results: {:?}", &large_cumsum.to_vec1::<f32>()?[..5]);
println!("  Last few results: {:?}", &large_cumsum.to_vec1::<f32>()?[size-5..]);

// Mixed positive/negative numbers
let mixed = Tensor::new(&[1.0f32, -2.0, 3.0, -4.0, 5.0, -6.0], &device)?;
let mixed_cumsum = mixed.cumsum(D::Minus1)?;
println!("\nMixed sign cumsum:");
println!("  Input: {:?}", mixed.to_vec1::<f32>()?);
println!("  Cumsum: {:?}", mixed_cumsum.to_vec1::<f32>()?);

// 3D tensor scan
let tensor_3d = Tensor::new(&[[[1.0f32, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]], &device)?;
println!("\n3D tensor scan along last dimension:");
let scan_3d = tensor_3d.cumsum(D::Minus1)?;
println!("  Shape: {:?}", scan_3d.shape());
println!("  Result: {:?}", scan_3d.to_vec3::<f32>()?);

// Edge case: single element
let single = Tensor::new(&[42.0f32], &device)?;
let single_cumsum = single.cumsum(D::Minus1)?;
println!("\nSingle element cumsum: {:?}", single_cumsum.to_vec1::<f32>()?);

println!("\n✓ Advanced scan features demonstrated successfully!");

Large tensor scan performance:
  Size: 10000 elements
  Time: 895.069028ms
  First few results: [1.0, 2.0, 3.0, 4.0, 5.0]
  Last few results: [9996.0, 9997.0, 9998.0, 9999.0, 10000.0]

Mixed sign cumsum:
  Input: [1.0, -2.0, 3.0, -4.0, 5.0, -6.0]
  Cumsum: [1.0, -1.0, 2.0, -2.0, 3.0, -3.0]

3D tensor scan along last dimension:
  Shape: [2, 2, 2]
  Result: [[[1.0, 3.0], [3.0, 7.0]], [[5.0, 11.0], [7.0, 15.0]]]

Single element cumsum: [42.0]

✓ Advanced scan features demonstrated successfully!
  Size: 10000 elements
  Time: 895.069028ms
  First few results: [1.0, 2.0, 3.0, 4.0, 5.0]
  Last few results: [9996.0, 9997.0, 9998.0, 9999.0, 10000.0]

Mixed sign cumsum:
  Input: [1.0, -2.0, 3.0, -4.0, 5.0, -6.0]
  Cumsum: [1.0, -1.0, 2.0, -2.0, 3.0, -3.0]

3D tensor scan along last dimension:
  Shape: [2, 2, 2]
  Result: [[[1.0, 3.0], [3.0, 7.0]], [[5.0, 11.0], [7.0, 15.0]]]

Single element cumsum: [42.0]

✓ Advanced scan features demonstrated successfully!


## Candle Scan Operations API Reference

Candle provides several scan operation methods optimized for different use cases:

### Core Methods

```rust
use candle_core::{Tensor, Device, D};

let tensor = Tensor::new(&[1.0, 2.0, 3.0, 4.0], &device)?;

// Cumulative sum (most common)
let cumsum = tensor.cumsum(D::Minus1)?;           // [1.0, 3.0, 6.0, 10.0]

// Inclusive scan (equivalent to cumsum, CUDA-optimized)
let inclusive = tensor.inclusive_scan(D::Minus1)?; // [1.0, 3.0, 6.0, 10.0]

// Exclusive scan (zero-padded prefix)
let exclusive = tensor.exclusive_scan(D::Minus1)?; // [0.0, 1.0, 3.0, 6.0]
```

### Multi-dimensional Scans

```rust
let matrix = Tensor::new(&[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], &device)?;

// Scan along rows (dimension 1)
let row_scan = matrix.cumsum(1)?;

// Scan along columns (dimension 0)  
let col_scan = matrix.cumsum(0)?;
```

### Performance Features

- **GPU Acceleration**: CUDA-optimized scan kernels when available
- **Work-Efficient**: O(n) time complexity with minimal memory overhead
- **Multi-dimensional**: Efficient scanning along any tensor dimension
- **Type Support**: Works with f32, f64, i32, i64, and other numeric types
- **Memory Efficient**: In-place optimizations where possible

### Key Differences from Standard Libraries

1. **Tensor-Native**: Operates on Candle tensors with shape preservation
2. **GPU-Accelerated**: Automatic CUDA optimization for large tensors  
3. **Dimension-Aware**: Scan along any dimension with intuitive syntax
4. **Type-Flexible**: Supports multiple numeric data types natively

This demonstrates the complete Candle scan operations API for prefix-sum computations.

In [5]:
// Quick 2D Scan Operations Demo
// Testing multi-dimensional scan functionality

// Create a simple 2x3 matrix for demonstration
let matrix_2d = Tensor::new(&[[1.0f32, 2.0, 3.0], [4.0, 5.0, 6.0]], &device)?;
println!("2D Matrix: {:?}", matrix_2d.to_vec2::<f32>()?);

// Row-wise cumsum (scan along columns within each row)
let row_cumsum = matrix_2d.cumsum(1)?;
println!("Row-wise cumsum: {:?}", row_cumsum.to_vec2::<f32>()?);

// Column-wise cumsum (scan along rows within each column)
let col_cumsum = matrix_2d.cumsum(0)?;
println!("Column-wise cumsum: {:?}", col_cumsum.to_vec2::<f32>()?);

// Test exclusive scan in 2D
let exclusive_2d = matrix_2d.exclusive_scan(1)?;
println!("2D exclusive scan: {:?}", exclusive_2d.to_vec2::<f32>()?);

println!("\n🎯 2D scan operations work perfectly!");
println!("   Row cumsum: [[1,3,6], [4,9,15]] - cumulative within rows");
println!("   Col cumsum: [[1,2,3], [5,7,9]] - cumulative down columns");

2D Matrix: [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
Row-wise cumsum: [[1.0, 3.0, 6.0], [4.0, 9.0, 15.0]]
Column-wise cumsum: [[1.0, 2.0, 3.0], [5.0, 7.0, 9.0]]
2D exclusive scan: [[0.0, 1.0, 3.0], [0.0, 4.0, 9.0]]

🎯 2D scan operations work perfectly!
   Row cumsum: [[1,3,6], [4,9,15]] - cumulative within rows
   Col cumsum: [[1,2,3], [5,7,9]] - cumulative down columns
Row-wise cumsum: [[1.0, 3.0, 6.0], [4.0, 9.0, 15.0]]
Column-wise cumsum: [[1.0, 2.0, 3.0], [5.0, 7.0, 9.0]]
2D exclusive scan: [[0.0, 1.0, 3.0], [0.0, 4.0, 9.0]]

🎯 2D scan operations work perfectly!
   Row cumsum: [[1,3,6], [4,9,15]] - cumulative within rows
   Col cumsum: [[1,2,3], [5,7,9]] - cumulative down columns
